# Concise Evidence-Backed Project State Report

Prepared on 2026-06-07. Filename uses the requested `003_2026-07-02_` prefix.

Scope: project-wide experiment progression across `docs/experiments` and `outputs/experiments`, including the numbered archive through completed run `870`. Run `871` currently contains only empty `data/` and `figures/` directories, so it is not counted as completed evidence.


## Executive State

The project is no longer a single FWI script. It is a controlled synthetic GPR-FDTD-FWI research system with separate but connected workflows for detection, single-rebar refinement, same-depth multi-rebar refinement, close-spacing stress tests, variable-depth/variable-radius coordinate recovery, source-shape/ringdown stress, and confidence reporting.

The strongest current claim is not universal global inversion. The strongest claim is narrower and better supported: in controlled 2D synthetic settings, the mature workflows can recover rebar location and radius for several single- and multi-rebar scenarios under noise and source mismatch, while reporting ambiguity intervals when the objective does not support a high-confidence scalar radius.

The current best multi-rebar frontier is variable-depth/variable-radius recovery with source mismatch, 10% noise, fitted source ringdown, Tx/Rx acquisition policy checks, and target-specific source-count policies. The current best tight-spacing frontier is a separate close-spacing target-focused branch, not proof that the fitted-ringdown variable-depth workflow automatically handles tangent spacing.


## Major Milestone Route

| Station | Evidence | Capability demonstrated | What changed | Why it mattered | Cumulative status |
| --- | --- | --- | --- | --- | --- |
| Solver foundation | Trackers `01`-`10`; especially `04_geometry_inversion`, `05_gpu_cpml` | GPU FDTD, adjoint validation, GPU CPML, first 3-rebar geometry inversion. Experiment `04` recovered 3 same-depth bars with <3 mm x error and <1 mm radius error. | Moved from forward simulation/pixel inversion toward geometric inversion. | Proved the simulator and parameterized inversion could recover rebar geometry. | Superseded as a workflow, but still the technical base. |
| Single-rebar exact/noise recovery | Trackers `11`-`14`; outputs `001`-`023` | One rebar x/z/r recovered with staged search and grid polish, including controlled noise up to 10%. | Replaced broad continuous optimization with deterministic local radius polish. | Exposed depth/radius coupling and hard-grid quantization. | Included later only inside local-basin workflows. Raw Powell radius was superseded. |
| Objective/source triage | Trackers `15`-`31`; outputs `024`-`058` | Tested trace-shift, bandwidth, W2/OT, material tradeoff, and wavelet mismatch. Source amplitude/time/frequency profiling fixed tested source-mismatch radius failures. | The project stopped treating wrong source shape as a minor nuisance. | This became essential for field-like robustness. | Fixed-source LS, W2 final radius, and free material radius explanations were rejected or demoted. |
| Robust single-rebar profiled polish | Trackers `32`-`33`; outputs `059`-`062` | Source-profiled polish passed exact, noise, source mismatch, seed replication, and wider local x/z/r windows. | Turned diagnostic source profiling into a reusable production-style local runner. | Established the mature single-rebar local estimator. | Still assumes a good local window; does not by itself detect the target. |
| Multi-rebar local profiling | Trackers `34`-`41`; outputs `063`-`080` | Three same-depth rebars: common radius, per-target radius, local x/z/r one target at a time. Stage 6 recovered 24/24 truth rows, but 22/24 were weak confidence. | Added top-k, confidence labels, and ambiguity intervals. | Prevented false precision when correct candidates barely won. | Includes earlier radius profiling only when other targets are fixed or locally controlled. |
| Sequential coordinate optimizer | Trackers `42`-`44`; outputs `081`-`098` | Bounded all-target coordinate optimizer. Compact 4-seed 10% noise/source-mismatch runs recovered all three bars; 2 mm seed-offset stress required guarded revisit. | Moved from isolated target profiles to sequential all-target state updates. | Showed edge high-radius branches are recoverable but must be guarded. | Main same-depth multi-rebar optimizer, but still bounded/local rather than global. |
| Detection-to-FWI pipeline | Tracker `47`; outputs `107`-`134` | Hyperbola detector seeded x/z windows. Single-rebar detector hit 48/48 nominal and 48/48 source-mismatch scenarios. Packaged detector -> 2 mm screen -> 1 mm polish recovered x/z/r. | Added a seed layer before FWI refinement. | Made single-rebar recovery closer to an end-to-end workflow. | Detector estimates x/z only; radius remains a source-profiled FWI/refinement output. |
| Shallow small-radius ambiguity | Outputs `121`-`134`; aggregate `127`, `129`, `134` | Shallow `z=70 mm`, `r=4 mm` cases recovered the point radius under source mismatch/noise but with weak/broad intervals. Subcell and equal multifrequency did not solve it. | Shifted reporting from scalar radius to interval when margin is weak. | Prevented overclaiming high precision for hard shallow/small bars. | Not superseded; remains a known limitation. |
| Variable-radius close spacing | Outputs `216`-`419`; handoff tracker `48` | Same-depth variable radii `[5,6,8]`; close-spacing and acquisition sweeps. Close14 target-focused branch under Tx/Rx=50 reached a clean noise boundary near `19.642333984375%` RMS. | Explored lateral separation, Tx/Rx, source count, and noise boundaries. | Demonstrated separation/sizing under very tight target-focused conditions. | Strong branch, but not proof of general all-target variable-depth capability. |
| Variable-depth/variable-radius coordinate recovery | Trackers `54`-`66`; outputs `451`-`533`; key aggregate `498`, `532`, `533` | Combined `x=[150,250,350]`, `z=[80,100,120]`, `r=[5,6,8]`; detector assignment found physical seeds; staged coordinate path reached exact truth across seeds. Tx/Rx=50 aggregate had 12/12 truth rows, zero x/z ambiguity, max radius ambiguity 0.25 mm. | Combined variable depth and variable radius while keeping staged guardrails. | This is the strongest mature multi-rebar workflow before fitted ringdown. | Stronger than prior branches, but not a global all-parameter solve. Base remains update objective; `veryhigh` is reporting diagnostic. |
| Fitted ringdown and target-specific policy | Trackers `273`-`403`; outputs `740`-`870`; key summaries `808`, `828`, `865`, `870` | Added fitted source ringdown, Tx/Rx=60, and target-specific source-count policies. Ringdown035 `8/9/9` passed 9/9 exact/moderate. Ringdown0459375 transferred but near cutoff. Ringdown050 passes full policy for seeds 13/89/34; seed21 needs a lower practical threshold; run `870` shows seed55 target0 passes with low reserve. | The project moved from point recovery to stress-policy characterization. | Current frontier: exact rows matter, but reserve above cutoff is now the limiting evidence. | Does not automatically include shallow r4, close14 tangent spacing, or field-data claims. |


## Current Best Workflows

| Use case | Mature workflow/scripts | Confirmed capability | Important guardrail |
| --- | --- | --- | --- |
| Single rebar, detector-seeded | `run_rebar_detection_pipeline.py`, `run_detection_seeded_two_stage_refinement.py` | Detect x/z seed, coarse 2 mm screen, final 1 mm source-profiled radius polish. Non-default `z=110 mm`, `r=8 mm` mismatch/noise case recovered exact x/z/r with strong margin in output `120`. | Shallow `r=4 mm` cases recover point truth but require radius intervals. |
| Single rebar, local polish only | `run_single_rebar_source_profiled_polish.py`, `run_single_rebar_source_profiled_replication.py` | Robust local x/z/r radius selection under noise and source mismatch in outputs `057`-`062`. | Assumes x/z is already in a good local basin. |
| Same-depth multi-rebar local/coordinate | `run_multi_rebar_local_geometry_profile.py`, `run_multi_rebar_coordinate_optimizer.py`, `run_candidate_confidence_report.py` | Three bars at same depth recovered across local and compact coordinate runs; guarded revisit fixes wider-offset edge high-radius branches. | Confidence/ambiguity fields are mandatory, especially for edge targets. |
| Variable-radius close spacing | `run_multi_rebar_coordinate_optimizer.py`, `run_variable_radius_staged_pipeline_summary.py`, aggregate scripts | Target-focused variable-radius/close-spacing recovery, including close14 Tx/Rx=50 noise-boundary evidence. | Branch-specific; not a universal multi-target close-spacing solver. |
| Variable-depth/variable-radius | `run_detection_assignment_report.py`, `run_assigned_coordinate_command_report.py`, `run_multi_rebar_coordinate_optimizer.py`, `run_coordinate_confidence_aggregate.py`, `run_coordinate_objective_diagnostic_report.py` | Exact staged recovery for `[5,6,8]` mm radii at `[80,100,120]` mm depths under source mismatch and 10% noise, with Tx/Rx/acquisition/objective diagnostics. | Base objective is the production update rule; diagnostic objectives are not automatically update objectives. |
| Fitted-ringdown stress policy | `run_multi_rebar_coordinate_optimizer.py`, `run_cross_seed_fitted_ringdown_summary.py`, `run_cross_condition_fitted_ringdown_summary.py`, target-specific summary scripts | Target-specific source-count/ringdown policy evidence through output `870`. | Near-cutoff margins matter; seed21 is a documented exception at full ringdown050. |


## Strongest Demonstrable Results

1. Variable-depth/variable-radius staged coordinate recovery: outputs `451`-`533`, especially aggregate `498`, show 12/12 truth-geometry rows for Tx/Rx=50 final-state checks across seeds 13, 34, and 55, with zero x/z ambiguity and max radius ambiguity width 0.25 mm.

2. Cross-condition objective confidence: output `532` reports 27/27 exact rows across selected non-ringdown and fitted-ringdown rows. `veryhigh` improves reporting margins and collapses ambiguity in that package, but output `533` explicitly keeps `base` as the production update objective.

3. Fitted-ringdown target policy: output `808` shows ringdown035 target-specific Tx/Rx=60 policy exact/moderate across 3 seeds and 3 targets. Output `828` shows ringdown0459375 transfers across three seeds but with the limiting seed13 target1 row barely above cutoff. Output `865` shows full ringdown050 passes for seeds 13, 89, and 34, while seed21 requires a practical lower threshold. Output `870` adds seed55 target0 as a full-ringdown050 pass with low reserve.

4. Tight-spacing target-focused branch: output `418` reports a close14, Tx/Rx=50, target-focused noise boundary with promoted clean endpoint `19.642333984375%` RMS and final ambiguous upper at `19.642372131347656%` RMS. This is strong evidence for that branch, but it is not the same as a full all-target global close-spacing workflow.


## Confirmed, Partial, and Unresolved Capabilities

| Capability | Status | Evidence and caveat |
| --- | --- | --- |
| Single-rebar detection of x/z seeds | Confirmed in controlled synthetic 2D | Detector benchmark outputs `112` and `113`: 48/48 hit rate for nominal and source-mismatch matrices. |
| Single-rebar x/z/r refinement | Confirmed for tested synthetic scenarios | Outputs `118`-`120` and source-profiled local runs. Shallow r4 is point-correct but weak-confidence. |
| Source amplitude/time/frequency profiling | Confirmed as required | Trackers `26`, `31`, outputs `052`-`060`: raw fixed-source failures corrected by profiling. |
| Multi-rebar same-depth local sizing | Confirmed with weak margins | Outputs `063`-`080`: all tested targets recover truth, but many confidence labels are weak. |
| Bounded all-target coordinate optimization | Confirmed in tested synthetic windows | Outputs `081`-`098`: compact cases recover truth; 2 mm offset requires guarded revisit. |
| Variable-radius same-depth close spacing | Partially confirmed | Outputs `216`-`419`: strong target-focused evidence and staged summaries, but not a universal all-target solver. |
| Variable-depth/variable-radius workflow | Confirmed for staged synthetic cases | Outputs `451`-`533`, `740`-`870`: strongest current branch, with source/ringdown/acquisition policy caveats. |
| Global all-parameter inversion | Unresolved/not claimed | Existing mature workflows are bounded and staged; broad all-parameter commands remain deferred. |
| Field or lab data | Unresolved/not claimed | All major claims are controlled synthetic 2D evidence. |
| 3D extension | Unresolved/not claimed | Current project is 2D TMz FDTD/FWI. |


## Superseded or Demoted Approaches

- Pixel-wise broad FWI: useful for theory and validation, but not the mature rebar workflow.
- Raw Powell or continuous radius-only optimization: superseded by grid/local profiling and top-k margin reporting.
- Fixed-source least squares for radius under mismatch: superseded by source amplitude/time/frequency/ringdown profiling.
- W2/optimal transport as final radius objective: rejected for this problem because it flattened radius-sensitive amplitude information.
- PEBDD/progressive bandwidth as final radius selector: useful as a seed/basin idea, not the final radius decision.
- Free material inversion as the default radius remedy: material sweeps did not explain the main radius ambiguity and can hide geometry error.
- Uniform source-count/acquisition policies: later fitted-ringdown runs show target-specific policies are necessary.


## Cumulative Inclusion Check

Later milestones do not automatically subsume every earlier capability.

- The fitted-ringdown variable-depth/radius workflow includes source-profiled FWI concepts and coordinate confidence reporting, but it does not prove shallow single-rebar r4 high-confidence sizing.
- The variable-depth/radius workflow does not prove close14 tangent spacing. The close14 evidence is a separate target-focused branch.
- The detector-to-FWI single-rebar workflow really does include earlier local radius polish after detection, but the detector stage itself only provides x/z windows.
- The coordinate optimizer includes the local profiling machinery, but only inside bounded candidate windows and guarded update rules. It is not a global unconstrained optimizer.
- Diagnostic objectives such as `veryhigh` or `late_high` improve reporting in specific packages, but output `533` explicitly rejects global promotion without targeted re-testing.


## Current Limitations and Ambiguities

The remaining limitations are mostly about confidence, generality, and transfer.

- Radius confidence can remain weak even when the point estimate is correct. This appears for shallow/small bars, edge targets, source-shape center cases, and near-threshold fitted-ringdown policies.
- The recurring physical ambiguity is depth/radius coupling: a slightly deeper, larger bar can closely mimic the true response.
- Receiver/grid quantization affects Tx/Rx threshold interpretation. Nearest-grid and linear receiver sampling branches showed abrupt or seed-sensitive confidence changes.
- Source-count escalation is not monotonic enough to be treated as a general fix. Some targets need more sources; others weaken under the wrong source layout.
- Field-like source shape is only partially handled. Ringdown/fitted source branches are strong synthetic guardrails, not lab-calibrated source models.
- No field/lab validation, no 3D validation, and no broad coupled all-target/all-parameter global search have been demonstrated.


## Evidence Index

| Evidence topic | Primary files/runs |
| --- | --- |
| Archive health and post-crash run inflation | `docs/experiments/272_experiment_archive_health_report_current.md`, `outputs/experiments/739_experiment_archive_health_report_current` |
| Single-rebar source-profiled polish | `docs/experiments/31_source_profiled_radius_polish.md`, outputs `057`-`060` |
| Single-rebar detector-to-refinement | `docs/experiments/47_detection_to_fwi_pipeline.md`, outputs `107`-`134` |
| Multi-rebar local confidence | `docs/experiments/36_multi_rebar_confidence_reporting.md`, `40_stage6_all_target_confidence_synthesis.md`, outputs `070`, `079`, `080` |
| Coordinate optimizer | `docs/experiments/42_reporting_first_coordinate_optimizer.md`, `43_coordinate_optimizer_noise_replication.md`, `44_coordinate_optimizer_seed_offset_stress.md`, outputs `081`-`098` |
| Variable-radius close spacing | outputs `216`-`419`, especially `257`, `327`, `335`, `418`, and tracker `48_research_handoff_matrix.md` |
| Variable-depth/radius staged coordinate | `docs/experiments/54`-`66`, outputs `451`-`533`, especially `498`, `532`, `533` |
| Fitted-ringdown policy frontier | trackers `273`-`403`, outputs `740`-`870`, especially `808`, `828`, `865`, `870` |
